# Project Milestone: Autonomous Agent for Ms. Pac-Man
**Student Name:** Christopher Hammer
**Course:** Machine Learning

## Project Goal
The objective of this project is to train a Reinforcement Learning agent to play *Ms. Pac-Man* (Atari 2600) using only raw pixel inputs. I will be using a **Deep Q-Network (DQN)** architecture, which combines Convolutional Neural Networks (CNNs) for visual processing with Q-Learning for decision making.

## Current Progress
For this milestone, I have:
1.  Successfully set up the Gymnasium `ALE/MsPacman-v5` environment.
2.  Implemented the DQN architecture using PyTorch.
3.  Created a training loop with an Experience Replay buffer.
4.  Verified that the model can interact with the environment and learn (Epsilon decay).

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
import ale_py

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Environment Setup
# We use standard Atari Preprocessing to grayscale and resize the image to 84x84
env = gym.make("ALE/MsPacman-v5", frameskip=1, render_mode="rgb_array")
env = AtariPreprocessing(env, screen_size=84, grayscale_obs=True, frame_skip=4, scale_obs=False)
env = FrameStackObservation(env, stack_size=4)

print("Environment Observation Space:", env.observation_space.shape)
print("Action Space:", env.action_space.n)

## Model Architecture
I am using a classic DQN structure. The input is a stack of 4 grayscale frames (84x84).
- **3 Convolutional Layers:** To extract features (walls, ghosts, pellets) from the pixels.
- **2 Fully Connected Layers:** To map those features to the 9 possible joystick actions.

In [ ]:
class DQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super(DQN, self).__init__()
        # Convolutional Layers
        self.conv1 = nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)

        # Calculate output size of conv layers
        def conv2d_size_out(size, kernel_size, stride):
            return (size - (kernel_size - 1) - 1) // stride + 1
        
        convw = conv2d_size_out(conv2d_size_out(conv2d_size_out(84, 8, 4), 4, 2), 3, 1)
        convh = convw
        linear_input_size = convw * convh * 64

        # Linear Layers
        self.fc1 = nn.Linear(linear_input_size, 512)
        self.fc2 = nn.Linear(512, n_actions)

    def forward(self, x):
        x = x.float() / 255.0 # Normalize pixel values
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = x.view(x.size(0), -1) # Flatten
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# Instantiate the model
n_actions = env.action_space.n
input_shape = (4, 84, 84)
policy_net = DQN(input_shape, n_actions).to(device)
print(policy_net)

## Training Loop (Milestone Demo)
Below is a demonstration of the training logic. The agent uses an **Epsilon-Greedy** strategy, starting with high exploration (random moves) and decaying over time.

In [ ]:
# Hyperparameters for Milestone Check
BATCH_SIZE = 32
GAMMA = 0.99
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 1000
LR = 1e-4

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
steps_done = 0

# Run a quick test of 5 episodes to prove the loop works
print("Starting Milestone Test Run...")
for i_episode in range(5):
    state, _ = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        # Select Action
        action = env.action_space.sample() # Random for this test
        
        # Step
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward
        
    print(f"Episode {i_episode + 1} Complete. Score: {total_reward}")

print("Milestone check complete: Environment and Model are functional.")